<a href="https://colab.research.google.com/github/PonVo02/E-Wallet-Payment-Performance-Analysis-Python/blob/main/E_Wallet_Payment_Performance_Analysis_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA + Data Wrangling


  # **Part I: EDA - Exploratory Data Analysis**


## **1. Dataset: payment_enriched**

The `payment_enriched` dataset is created by merging:
- `payment_report.csv` : monthly payment volume by product
- `product.csv`: which contains product metadata such as (category, team ownership)

Each record represents the payment volume of a specific product in a given month and source.


In [70]:
import pandas as pd

payment = pd.read_csv('/content/payment_report.csv')
product = pd.read_csv('/content/product.csv')

payment_enriched = payment.merge(
    product,
    on = 'product_id',
    how = 'left',
    validate = 'm:1')

----
### **1.1 Dataset Overview**
Check the overall data

In [71]:
payment_enriched.head()

,report_month,payment_group,product_id,source_id,volume,category,team_own
0,2023-01,payment,12,45,624110375,PXXXXXT,ASD
1,2023-01,payment,17,45,335715113,PXXXXXB,ASD
2,2023-01,payment,18,45,737784466,PXXXXXB,ASD
3,2023-01,payment,19,45,120963069,PXXXXXM2,ASD
4,2023-01,payment,20,45,319653158,PXXXXXB,ASD


In [72]:
payment_enriched.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   report_month   919 non-null    object
 1   payment_group  919 non-null    object
 2   product_id     919 non-null    int64 
 3   source_id      919 non-null    int64 
 4   volume         919 non-null    int64 
 5   category       897 non-null    object
 6   team_own       897 non-null    object
dtypes: int64(3), object(4)
memory usage: 50.4+ KB


### Incorrect data types: `payment_enriched`

**Observed data types:**
- `report_month`: object (string – year-month format)
- `payment_group`: object
- `category`, `team_own`: object
- `product_id`, `source_id`: int64 (identifiers)
- `volume`: int64 (numeric metric)

**Assessment:**
- All data types are consistent with their business meaning.
- No type conversion is required at this stage.

-> Next step: No action

----
### **1.2 Missing Data Analysis**


In [73]:
payment_enriched.isna().sum()

,0
report_month,0
payment_group,0
product_id,0
source_id,0
volume,0
category,22
team_own,22


In [74]:
payment_enriched[payment_enriched['category'].isna()][['product_id']].head()

,product_id
190,10033
192,1976
193,1976
194,1976
195,1976


In [75]:
(payment_enriched['category'].isna() & payment_enriched['team_own'].isna()).sum()

np.int64(22)

**Missing data:**
- `category`: 22 rows missing
- `team_own`: 22 rows missing
- Missing values occur together on the same records.

**Interpretation:**
Because `payment_enriched` is created using a LEFT JOIN on `product_id`,
these missing values likely come from `product_id` that do not exist in `product.csv`
(unmatched product metadata).

-> Next step: No action

----
### **1.3 Duplicate Analysis**

In [76]:
# check dulicates based on business key

payment_enriched.duplicated(
    subset = ['report_month', 'product_id', 'source_id']).sum()

np.int64(5)

In [77]:
# Inspect duplicated records
payment_enriched[
    payment_enriched.duplicated(
        subset=['report_month', 'product_id', 'source_id'],
        keep = False)
].sort_values(['product_id', 'report_month'])

,report_month,payment_group,product_id,source_id,volume,category,team_own
194,2023-01,refund,1976,39,443387276,NaN,NaN
195,2023-01,refund,1976,39,111281678,NaN,NaN
409,2023-02,refund,1976,39,111500,NaN,NaN
412,2023-02,refund,1976,39,910819764,NaN,NaN
413,2023-02,refund,1976,39,2675894726,NaN,NaN
636,2023-03,refund,1976,39,1660931261,NaN,NaN
637,2023-03,refund,1976,39,4731275843,NaN,NaN
917,2023-04,refund,1976,39,1905435543,NaN,NaN
918,2023-04,refund,1976,39,3679922071,NaN,NaN


### Duplicate analysis: `payment_enriched`

**Key checked:** (`report_month`, `product_id`, `source_id`)

**Findings:**
- 5 duplicated records detected.
- Duplicates are mainly related to `refund` payment_group.
- Volumes are different across duplicated rows, indicating valid business events
  rather than data duplication errors.

**Interpretation:**
- Multiple payment/refund records for the same product and month are possible
  in real payment operations.

-> Next step: No action

----
### **1.4 Incorrect Data Types**

In [78]:
payment_enriched.dtypes

,0
report_month,object
payment_group,object
product_id,int64
source_id,int64
volume,int64
category,object
team_own,object


**Incorrect data types:**
- None detected. All data is consistent with business sense.

-> Next step: No action

----
### **1.5 Numerical Summary**

In [79]:
payment_enriched['volume'].describe()

,volume
count,9.190000e+02
mean,1.978574e+08
std,8.367595e+08
min,5.500000e+03
25%,1.250000e+06
50%,7.982015e+06
75%,5.447599e+07
max,1.383171e+10


**Incorrect values:**
- No negative or zero values detected in `volume`.

**Observations:**
- Distribution is right-skewed with a small number of extremely large values.
- Median is much lower than mean, indicating outliers influence the mean.

-> Next step: No action

----
## **EDA Summary – `payment_enriched`**

- **Incorrect data types:** None → Next step: No action  
- **Incorrect values:** No abnormal values in `volume` → Next step: No action  
- **Missing data:** `category` (22), `team_own` (22) due to unmatched product metadata after merge → Next step: No action  
- **Duplicates:** 5 rows based on key (`report_month`, `product_id`, `source_id`) but likely business-valid → Next step: No action  

Conclusion: `payment_enriched` is clean and ready for further analysis.


-----------

## **2. Dataset: transations**
The transactions dataset contains detailed transaction-level data generated by users and the system within the e-wallet platform.

Each record represents a single transaction event, including:
- Transaction amount
- Transaction type
- Sender and receiver
- Transaction status
- Timestamp of execution

This dataset helps analyze transaction flows, user behavior, and system operations in the e-wallet context.

-----

### **2.1 Dataset Overview**


In [80]:
df = pd.read_csv('/content/transactions.csv')

df.head()

,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,extra_info,timeStamp
0,3002692434,5,100000,24,1,10199794.0,199794.0,NaN,1682932054455
1,3002692437,305,20000,2,1,14022211.0,14022211.0,NaN,1682932054912
2,3001960110,7255,48605,22,1,NaN,10530940.0,NaN,1682932055000
3,3002680710,2270,1500000,2,1,10059206.0,59206.0,NaN,1682932055622
4,3002680713,2275,90000,2,1,10004711.0,4711.0,NaN,1682932056197


In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1324002 entries, 0 to 1324001
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   transaction_id  1324002 non-null  int64  
 1   merchant_id     1324002 non-null  int64  
 2   volume          1324002 non-null  int64  
 3   transType       1324002 non-null  int64  
 4   transStatus     1324002 non-null  int64  
 5   sender_id       1274943 non-null  float64
 6   receiver_id     1159207 non-null  float64
 7   extra_info      6095 non-null     object 
 8   timeStamp       1324002 non-null  int64  
dtypes: float64(2), int64(6), object(1)
memory usage: 90.9+ MB


**Observations:**
- The dataset contains **1,324,002 rows** and **9 columns**.
- Each row represents a **single transaction event** in the e-wallet system.
- Data includes transaction amount, transaction type, sender/receiver, status, and timestamp.

-----

### **2.2 Data Types Analysis**

In [82]:
df.dtypes

,0
transaction_id,int64
merchant_id,int64
volume,int64
transType,int64
transStatus,int64
sender_id,float64
receiver_id,float64
extra_info,object
timeStamp,int64


In [83]:
# timestamp to datetime

df['timeStamp'] = pd.to_datetime(df['timeStamp'], unit = 'ms')

df.head()


,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,extra_info,timeStamp
0,3002692434,5,100000,24,1,10199794.0,199794.0,NaN,2023-05-01 09:07:34.455
1,3002692437,305,20000,2,1,14022211.0,14022211.0,NaN,2023-05-01 09:07:34.912
2,3001960110,7255,48605,22,1,NaN,10530940.0,NaN,2023-05-01 09:07:35.000
3,3002680710,2270,1500000,2,1,10059206.0,59206.0,NaN,2023-05-01 09:07:35.622
4,3002680713,2275,90000,2,1,10004711.0,4711.0,NaN,2023-05-01 09:07:36.197


### **Data Types**

- Numeric columns: `transaction_id`, `merchant_id`, `volume`,
  `transType`, `transStatus`
- Identifier columns: `sender_id`, `receiver_id`
- Text column: `extra_info`
- Datetime column: `timeStamp`

**Assessment:**
- Data types are consistent with transaction business meaning.
- `timeStamp` has been standardized to datetime format.

-> Next step: No action

-----

## **2.3 Missing Data**

In [84]:
df.isna().sum()

,0
transaction_id,0
merchant_id,0
volume,0
transType,0
transStatus,0
sender_id,49059
receiver_id,164795
extra_info,1317907
timeStamp,0


In [85]:
df[df['sender_id'].isna()].head()

,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,extra_info,timeStamp
2,3001960110,7255,48605,22,1,NaN,10530940.0,NaN,2023-05-01 09:07:35
11,3001888503,7255,1000000,22,1,NaN,14008575.0,NaN,2023-05-01 09:07:38
15,3001888506,7255,225000,22,1,NaN,1967742.0,NaN,2023-05-01 09:07:39
31,3001900413,7255,10000,22,1,NaN,4368.0,NaN,2023-05-01 09:07:44
54,3001936221,7255,100,22,1,NaN,4129699.0,NaN,2023-05-01 09:07:51


In [86]:
df[df['receiver_id'].isna()].head()

,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,extra_info,timeStamp
7,3002680722,6075,1000,2,1,10012978.0,NaN,NaN,2023-05-01 09:07:37.146
8,3002680725,2895,83158,2,1,10089635.0,NaN,NaN,2023-05-01 09:07:37.228
17,3002675016,5700,10000,2,1,21024640.0,NaN,NaN,2023-05-01 09:07:39.370
20,3002675025,7435,20000,2,1,10073374.0,NaN,NaN,2023-05-01 09:07:40.500
22,3002675028,3075,10300,2,1,38512397.0,NaN,NaN,2023-05-01 09:07:40.565


**Missing data detected:**
- `sender_id`: missing values
- `receiver_id`: missing values
- `extra_info`: mostly missing
- Other columns: no missing values

**Interpretation:**
- Missing `sender_id` or `receiver_id` is expected in system-related transactions
  such as top-ups, refunds, or cashback.
- `extra_info` has a high number of missing values and does not affect core analysis, so it is ignored.

-> Next step: No action

-----

## **2.4 Transaction Type**

In [87]:
df['transType'].value_counts()

,count
transType,
2,760783
8,342553
22,202383
30,14586
24,3677
14,10
12,10


In [88]:
df.groupby('transType')[['sender_id', 'receiver_id']].apply(lambda x: x.isna().sum())

,sender_id,receiver_id
transType,,
2,0,162208
8,0,0
12,0,0
14,0,0
22,34473,0
24,0,0
30,14586,2587


**Observations:**
- The dataset contains multiple transaction types encoded as numeric values.
- The most frequent transaction types are:
- 2: highest number of transactions
- 8 and 22: also appear frequently
- 12, 14, 24, 30: very rare and considered edge cases

**Missing sender & receiver analysis:**
- transType = 2: some transactions have missing receiver_id
- transType = 22: many transactions have missing sender_id
- Other transaction types show no missing sender or receiver

**Interpretation:**
- Missing sender_id or receiver_id depends on the transaction type.
- This behavior is expected for system-related transactions (e.g. top-up, refund).
- These missing values do not indicate data quality issues.

-> Next step: No action


----

## **2.5 Duplicate**

In [89]:
df['transaction_id'].duplicated().sum()

np.int64(28)

In [90]:
df[df['transaction_id'].duplicated(keep=False)].head(10)

,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,extra_info,timeStamp
149734,3000871152,75,193000,2,1,17027174.0,NaN,NaN,2023-05-02 00:50:17.608
149735,3000871152,75,193000,2,1,17027174.0,NaN,NaN,2023-05-02 00:50:17.608
149738,3000871161,2270,50000,2,1,10009352.0,9352.0,NaN,2023-05-02 00:50:17.770
149739,3000871161,2270,50000,2,1,10009352.0,9352.0,NaN,2023-05-02 00:50:17.770
149752,3000687030,2250,165000,8,1,10074844.0,38548300.0,NaN,2023-05-02 00:50:23.437
149753,3000687030,2250,165000,8,1,10074844.0,38548300.0,NaN,2023-05-02 00:50:23.437
149757,3000871242,2270,300000,2,1,35013832.0,35013832.0,NaN,2023-05-02 00:50:25.523
149758,3000871242,2270,300000,2,1,35013832.0,35013832.0,NaN,2023-05-02 00:50:25.523
149761,3000871251,5,200,22,1,10028007.0,10529190.0,NaN,2023-05-02 00:50:26.225
149762,3000871251,5,200,22,1,10028007.0,10529190.0,NaN,2023-05-02 00:50:26.225


**Key checked:** `transaction_id`

**Findings:**

- `transaction_id` contains 28 duplicated records.
- Duplicated rows are identical across all key fields
  (amount, sender, receiver, timestamp).
- This likely comes from system logging or retry behavior,
  not duplicated business transactions.

-> Next step: No action

-----

## **2.6 Numerical Summary**

In [91]:
df['volume'].describe()

,volume
count,1.324002e+06
mean,2.388059e+05
std,9.681009e+05
min,1.000000e+00
25%,1.000000e+04
50%,3.000000e+04
75%,1.000000e+05
max,7.869148e+07


### 2.6 Numerical Summary

**Observations:**
- Most transactions have small to medium volumes.
- The distribution is right-skewed with a few very large transactions.
- Mean is higher than median due to large-value transactions.

Interpretation:
- Median better represents a typical transaction value.
- Large volumes are expected in e-wallet data.

→ Next step: No action


----

## **EDA Summary – transactions**


- **Data types:** All columns have appropriate data types → No action  
- **Values:** `volume` contains only positive values → No action  
- **Missing data:**  
  Missing `sender_id` / `receiver_id` occurs in system-related transactions.  
  `extra_info` is mostly missing and not needed for analysis → No action  
- **Duplicates:**  
  28 duplicated `transaction_id` records detected, rows are identical → No action  

**Conclusion:**  
The `transactions` dataset is consistent and ready for further analysis.


-----

  # **Part II: Data Wrangling**


In [92]:
payment_enriched.head()

,report_month,payment_group,product_id,source_id,volume,category,team_own
0,2023-01,payment,12,45,624110375,PXXXXXT,ASD
1,2023-01,payment,17,45,335715113,PXXXXXB,ASD
2,2023-01,payment,18,45,737784466,PXXXXXB,ASD
3,2023-01,payment,19,45,120963069,PXXXXXM2,ASD
4,2023-01,payment,20,45,319653158,PXXXXXB,ASD


### **1. Top 3 product_ids with the highest volume.**

In [93]:
# 1. Top 3 product_ids with the highest volume.

top_3_products = (
    payment_enriched.groupby('product_id')['volume'].sum().sort_values(ascending=False).head(3)
)

print('Top 3 product_ids with the highest volume:')
print(top_3_products)

Top 3 product_ids with the highest volume:
product_id
1976    61797583647
429     14667676567
372     13713658515
Name: volume, dtype: int64


-----

### **2. Given that 1 product_id is only owed by 1 team, are there any abnormal products against this rule?**

In [94]:
# 2. Given that 1 product_id is only owed by 1 team, are there any abnormal products against this rule?

abnormal_products = (
    payment_enriched.groupby('product_id')['team_own']
    .nunique()
    .reset_index(name='team_count')
    .query('team_count > 1'))

if abnormal_products.empty:
  print('No abnormal products found.')
else:
  print('Abnormal products found:')
  print(abnormal_products)

No abnormal products found.


-----

### **3. Find the team has had the lowest performance (lowest volume) since Q2.2023. Find the category that contributes the least to that team.**

In [95]:
# 3. Find the team has had the lowest performance (lowest volume) since Q2.2023. Find the category that contributes the least to that team.

# filter Q2.2023
payment_enriched['report_month'] = pd.to_datetime(payment_enriched['report_month'])

payment_enriched_q2 = payment_enriched[payment_enriched['report_month'] >= '2023-04-01']

#lowest pfm
team_volume = (payment_enriched_q2.groupby('team_own')['volume']
                   .sum()
                   .sort_values())

team_lowest = team_volume.index[0]
team_lowest_volume = team_volume.iloc[0]

print(f'Team with the lowest performance: {team_lowest}')
print(f'Total volume: {team_lowest_volume}')

# Category contributing the least to that team

category_contributing = (
    payment_enriched_q2[payment_enriched_q2['team_own'] == team_lowest]
    .groupby('category')['volume']
    .sum()
    .sort_values()
    .head(1))

least_category = category_contributing.index[0]
least_category_volume = category_contributing.iloc[0]

print("Least-contributing category:", least_category)
print("Category volume:", least_category_volume)



Team with the lowest performance: APS
Total volume: 51141753
Least-contributing category: PXXXXXE
Category volume: 25232438


------

### **4. Find the contribution of source_ids of refund transactions (payment_group = 'refund'), what is the source_id with the highest contribution?**

In [96]:
# 4. Find the contribution of source_ids of refund transactions (payment_group = 'refund'), what is the source_id with the highest contribution?

# Filter refund transactions
rf_payment_enriched = payment_enriched[payment_enriched['payment_group'] == 'refund']

# Refund volume by source_id + contribution %
rf_contribution = (
    rf_payment_enriched.groupby('source_id')['volume']
    .sum()
    .sort_values(ascending=False)
    .reset_index(name='refund_volume')
)

rf_contribution["pct_contribute"] = rf_contribution['refund_volume'] / rf_contribution['refund_volume'].sum() * 100

top_source_id = rf_contribution.iloc[0]['source_id']
top_pct = rf_contribution.iloc[0]['pct_contribute']

print("Top refund source_id:", top_source_id)
print("Contribution (%):", top_pct)

rf_contribution

Top refund source_id: 38.0
Contribution (%): 59.10822495528634


,source_id,refund_volume,pct_contribute
0,38,36527454759,59.108225
1,39,16119059662,26.083641
2,37,9151069226,14.808134


-----


### **5. Define type of transactions ('transaction_type') for each row, given:**
- transType = 2 & merchant_id = 1205: Bank Transfer Transaction
- transType = 2 & merchant_id = 2260: Withdraw Money Transaction
- transType = 2 & merchant_id = 2270: Top Up Money Transaction
- transType = 2 & others merchant_id: Payment Transaction
- transType = 8, merchant_id = 2250: Transfer Money Transaction
- transType = 8 & others merchant_id: Split Bill Transaction
- Remained cases are invalid transactions

In [97]:
# tranTpe = 2 - Bank Transfer - Withdraw Money - Top Up Money
def transaction_type_m(x):
  if x['transType'] == 2:
    if x['merchant_id'] == 1205:
      return 'Bank Transfer'
    elif x['merchant_id'] == 2260:
      return 'Withdraw Money'
    elif x['merchant_id'] == 2270:
      return 'Top Up Money'
    else:
      return 'Payment'
  # trantype = 8 - Tranfers Money - Split Bill
  elif x['transType'] == 8:
    if x['merchant_id'] == 2250:
      return 'Transfer Money'
    else:
       return 'Split Bill'

  # remain
  else:
    return ' Invalid'

df['transaction_type'] = df.apply(transaction_type_m, axis=1)
df['transaction_type'].value_counts()

,count
transaction_type,
Payment,398677
Transfer Money,341177
Top Up Money,290502
Invalid,220666
Bank Transfer,37879
Withdraw Money,33725
Split Bill,1376


-----


### **6. Of each transaction type (excluding invalid transactions): find the number of transactions, volume, senders and receivers.**

In [98]:
# filter valid transactions
df_valid = df[df['transaction_type'] != 'Invalid']

# Number Transaction

number_transaction = (
    df_valid.groupby('transaction_type')['transaction_id'].count()
)

In [99]:
# Volume
volume_transaction = (
    df_valid.groupby('transaction_type')['volume'].sum()
)

volume_transaction

,volume
transaction_type,
Invalid,24659408387
Bank Transfer,50605806190
Payment,71851515181
Split Bill,4901464
Top Up Money,108606478829
Transfer Money,37033171492
Withdraw Money,23418181420


In [100]:
# senders vaf receivers

senders_receivers = (
    df_valid.groupby('transaction_type').agg(
        transactions = ('transaction_id', 'count'),
        total_volume = ('volume', 'sum'),
        unique_senders = ('sender_id', 'nunique'),
        unique_receivers = ('receiver_id','nunique')
    )
)

senders_receivers

,transactions,total_volume,unique_senders,unique_receivers
transaction_type,,,,
Invalid,220666,24659408387,2708,89716
Bank Transfer,37879,50605806190,23156,9271
Payment,398677,71851515181,139583,113298
Split Bill,1376,4901464,1323,572
Top Up Money,290502,108606478829,110409,110409
Transfer Money,341177,37033171492,39021,34585
Withdraw Money,33725,23418181420,24814,24814
